# Async Python para IA: asyncio, AsyncAnthropic y Rate Limiting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/python-para-ia/04-async-python-ia.ipynb)

asyncio, AsyncAnthropic y procesamiento paralelo de documentos con rate limiting.

Las llamadas a LLMs son lentas (1-30 segundos). El procesamiento síncrono de 100 documentos puede tardar horas. Con `asyncio` y `AsyncAnthropic` procesamos decenas de documentos en paralelo reduciendo el tiempo total en un orden de magnitud, sin superar los límites de la API.

In [ ]:
%pip install anthropic --quiet

import asyncio
import time
import random
import os
from typing import Optional

# Configuración para Jupyter (event loop ya activo)
import nest_asyncio
nest_asyncio.apply()

print('Entorno async listo.')

## 1. Async básico: gather vs secuencial

`asyncio.gather` ejecuta coroutines en paralelo dentro del mismo hilo. No paralelismo real (GIL), pero las esperas de I/O (red) se solapan.

In [ ]:
async def simular_llamada_api(doc_id: int, latencia: float = None) -> dict:
    """Simula una llamada a API con latencia variable."""
    lat = latencia or random.uniform(0.5, 2.0)
    await asyncio.sleep(lat)  # simula espera de red
    return {'id': doc_id, 'resultado': f'Resumen del documento {doc_id}', 'latencia': lat}

# Secuencial
async def procesar_secuencial(ids: list[int]) -> list:
    resultados = []
    for doc_id in ids:
        resultado = await simular_llamada_api(doc_id, latencia=0.3)
        resultados.append(resultado)
    return resultados

# Paralelo con gather
async def procesar_paralelo(ids: list[int]) -> list:
    tareas = [simular_llamada_api(doc_id, latencia=0.3) for doc_id in ids]
    return await asyncio.gather(*tareas)

ids = list(range(1, 11))  # 10 documentos

# Benchmark
t0 = time.time()
resultados_sec = asyncio.run(procesar_secuencial(ids))
t_sec = time.time() - t0

t0 = time.time()
resultados_par = asyncio.run(procesar_paralelo(ids))
t_par = time.time() - t0

print(f'Secuencial:  {t_sec:.2f}s  ({len(resultados_sec)} docs)')
print(f'Paralelo:    {t_par:.2f}s  ({len(resultados_par)} docs)')
print(f'Aceleración: {t_sec/t_par:.1f}x')

## 2. AsyncAnthropic: cliente asíncrono oficial

`anthropic.AsyncAnthropic` tiene la misma interfaz que el cliente síncrono pero todas las llamadas son coroutines.

In [ ]:
import anthropic

async def resumir_documento(cliente: anthropic.AsyncAnthropic,
                             texto: str, doc_id: int) -> dict:
    """Resume un documento usando AsyncAnthropic."""
    t0 = time.time()
    try:
        respuesta = await cliente.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=200,
            messages=[{
                'role': 'user',
                'content': f'Resume en 2 frases: {texto[:500]}'
            }]
        )
        return {
            'id': doc_id,
            'resumen': respuesta.content[0].text,
            'tokens_usados': respuesta.usage.input_tokens + respuesta.usage.output_tokens,
            'latencia': time.time() - t0
        }
    except Exception as e:
        return {'id': doc_id, 'error': str(e), 'latencia': time.time() - t0}

# Ejemplo de uso (requiere ANTHROPIC_API_KEY)
async def demo_async_anthropic():
    documentos = [
        'La inteligencia artificial es una rama de la informática que busca crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana.',
        'El aprendizaje automático es un subconjunto de la IA que permite a los sistemas aprender automáticamente y mejorar a partir de la experiencia.',
        'Las redes neuronales son modelos computacionales inspirados en el cerebro humano, con capas de neuronas artificiales interconectadas.'
    ]
    async with anthropic.AsyncAnthropic() as cliente:
        tareas = [resumir_documento(cliente, doc, i) for i, doc in enumerate(documentos)]
        resultados = await asyncio.gather(*tareas)
    return resultados

print('Función demo_async_anthropic() lista.')
print('Ejecutar: resultados = asyncio.run(demo_async_anthropic())')
print('(Requiere ANTHROPIC_API_KEY en el entorno)')

## 3. BatchProcessor con Semaphore

`asyncio.Semaphore` limita el número de tareas concurrentes. Sin él, lanzar 1000 tareas simultáneas agotaría el rate limit de la API en segundos.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Any

@dataclass
class ResultadoBatch:
    total: int = 0
    exitosos: int = 0
    fallidos: int = 0
    resultados: list = field(default_factory=list)
    tiempo_total: float = 0.0

class BatchProcessor:
    """
    Procesador de lotes con control de concurrencia.
    Garantiza que no más de `max_concurrente` tareas se ejecuten a la vez.
    """

    def __init__(self, max_concurrente: int = 5, mostrar_progreso: bool = True):
        self.max_concurrente = max_concurrente
        self.mostrar_progreso = mostrar_progreso

    async def procesar(self, items: list, fn: Callable) -> ResultadoBatch:
        """
        Procesa una lista de items aplicando fn a cada uno,
        con concurrencia máxima controlada por semáforo.
        """
        semaforo = asyncio.Semaphore(self.max_concurrente)
        resultado = ResultadoBatch(total=len(items))
        completados = 0

        async def procesar_uno(item, idx: int):
            nonlocal completados
            async with semaforo:  # bloquea si hay max_concurrente tareas activas
                try:
                    res = await fn(item)
                    resultado.resultados.append(res)
                    resultado.exitosos += 1
                except Exception as e:
                    resultado.resultados.append({'error': str(e), 'item': item})
                    resultado.fallidos += 1
                finally:
                    completados += 1
                    if self.mostrar_progreso and completados % max(1, len(items)//5) == 0:
                        pct = completados / len(items) * 100
                        print(f'  Progreso: {completados}/{len(items)} ({pct:.0f}%)')

        t0 = time.time()
        await asyncio.gather(*[procesar_uno(item, i) for i, item in enumerate(items)])
        resultado.tiempo_total = time.time() - t0
        return resultado

# Demo sin API real
async def demo_batch():
    async def tarea_simulada(item: str) -> dict:
        await asyncio.sleep(random.uniform(0.1, 0.3))
        return {'item': item, 'resultado': f'procesado_{item}'}

    documentos = [f'doc_{i}' for i in range(20)]
    processor = BatchProcessor(max_concurrente=5)

    resultado = await processor.procesar(documentos, tarea_simulada)
    print(f'\nTotal: {resultado.total} | Éxitos: {resultado.exitosos} | Fallos: {resultado.fallidos}')
    print(f'Tiempo: {resultado.tiempo_total:.2f}s')
    return resultado

resultado = asyncio.run(demo_batch())

## 4. Streaming asíncrono

El streaming recibe los tokens a medida que el modelo los genera, reduciendo la latencia percibida. Con `AsyncAnthropic` el streaming no bloquea el event loop.

In [ ]:
async def stream_respuesta(prompt: str) -> str:
    """
    Recibe tokens en streaming e imprime en tiempo real.
    Devuelve el texto completo al final.
    """
    async with anthropic.AsyncAnthropic() as cliente:
        texto_completo = []
        print('Respuesta: ', end='', flush=True)
        async with cliente.messages.stream(
            model='claude-haiku-4-5-20251001',
            max_tokens=300,
            messages=[{'role': 'user', 'content': prompt}]
        ) as stream:
            async for texto in stream.text_stream:
                print(texto, end='', flush=True)
                texto_completo.append(texto)
        print()  # nueva línea al final
        return ''.join(texto_completo)

# Uso (requiere API key):
# asyncio.run(stream_respuesta('Explica qué es asyncio en Python en 3 frases.'))

print('stream_respuesta() implementada.')
print('El streaming con AsyncAnthropic usa async with cliente.messages.stream()')

## 5. TokenBucket: rate limiting preciso

El **Token Bucket** es el algoritmo más preciso para rate limiting. Los tokens se añaden continuamente a una velocidad fija; cada llamada consume tokens. Si no hay suficientes, se espera.

In [ ]:
class TokenBucket:
    """
    Implementación de Token Bucket para rate limiting asíncrono.
    Garantiza que no se supere `tasa` operaciones por segundo en promedio.
    """

    def __init__(self, tasa: float, capacidad: float):
        """
        Args:
            tasa: tokens por segundo que se añaden al bucket
            capacidad: máximo de tokens almacenables (permite bursts)
        """
        self.tasa = tasa
        self.capacidad = capacidad
        self.tokens = capacidad  # empieza lleno
        self.ultimo_rellenado = time.monotonic()
        self._lock = asyncio.Lock()

    async def adquirir(self, tokens: float = 1.0):
        """Espera hasta que haya suficientes tokens disponibles."""
        async with self._lock:
            while True:
                ahora = time.monotonic()
                elapsed = ahora - self.ultimo_rellenado
                self.tokens = min(self.capacidad, self.tokens + elapsed * self.tasa)
                self.ultimo_rellenado = ahora

                if self.tokens >= tokens:
                    self.tokens -= tokens
                    return
                else:
                    # Calcular espera mínima
                    espera = (tokens - self.tokens) / self.tasa
                    await asyncio.sleep(espera)


# Demo: 10 req/s con burst de 5
async def demo_rate_limiting():
    bucket = TokenBucket(tasa=10.0, capacidad=5.0)
    tiempos = []

    async def llamada_con_rl(idx: int):
        await bucket.adquirir()
        tiempos.append(time.monotonic())

    t0 = time.monotonic()
    await asyncio.gather(*[llamada_con_rl(i) for i in range(15)])
    t_total = time.monotonic() - t0

    print(f'15 llamadas con rate limit 10 req/s + burst 5:')
    print(f'Tiempo total: {t_total:.2f}s')
    print(f'Tasa efectiva: {15/t_total:.1f} req/s')

    # Intervalos entre llamadas
    intervalos = [tiempos[i+1]-tiempos[i] for i in range(len(tiempos)-1)]
    print(f'Intervalo medio: {sum(intervalos)/len(intervalos)*1000:.0f}ms')

asyncio.run(demo_rate_limiting())

## 6. Benchmark completo: síncrono vs asíncrono

Medición rigurosa del speedup con distintos niveles de concurrencia y latencia simulada.

In [ ]:
import matplotlib.pyplot as plt

async def benchmark_concurrencia(n_docs: int = 30, latencia_simulada: float = 0.5):
    """Compara tiempo de procesamiento con distintos niveles de concurrencia."""

    async def tarea(item):
        await asyncio.sleep(latencia_simulada)
        return item

    docs = list(range(n_docs))
    resultados_tiempo = {}

    # Secuencial
    t0 = time.time()
    for doc in docs:
        await tarea(doc)
    resultados_tiempo['Secuencial\n(conc=1)'] = time.time() - t0

    # Paralelo con distintas concurrencias
    for conc in [5, 10, 20, n_docs]:
        processor = BatchProcessor(max_concurrente=conc, mostrar_progreso=False)
        t0 = time.time()
        await processor.procesar(docs, tarea)
        label = f'Async\n(conc={conc})' if conc < n_docs else f'Async\n(conc=all)'
        resultados_tiempo[label] = time.time() - t0

    return resultados_tiempo

resultados = asyncio.run(benchmark_concurrencia(n_docs=30, latencia_simulada=0.3))

# Visualización
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(resultados.keys())
tiempos = list(resultados.values())
colores = ['#e74c3c'] + ['#3498db'] * (len(labels) - 1)
barras = ax.bar(labels, tiempos, color=colores, edgecolor='white', linewidth=0.5)

for barra, t in zip(barras, tiempos):
    ax.text(barra.get_x() + barra.get_width()/2, t + 0.1,
            f'{t:.1f}s', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Tiempo total (segundos)')
ax.set_title(f'Benchmark: 30 documentos, latencia simulada 0.3s/doc')
ax.set_ylim(0, max(tiempos) * 1.2)

# Anotar speedup
t_seq = tiempos[0]
for i, t in enumerate(tiempos[1:], 1):
    ax.text(i, 0.5, f'{t_seq/t:.1f}x\nmás rápido',
            ha='center', color='white', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()
print(f'Speedup máximo: {tiempos[0]/min(tiempos[1:]):.1f}x')

## 7. Pipeline completo: async + rate limiting + batching

Combinamos todos los componentes en un pipeline listo para producción.

In [ ]:
class PipelineDocumentos:
    """
    Pipeline asíncrono completo para procesar documentos con Claude.
    Incluye rate limiting, control de concurrencia y manejo de errores.
    """

    def __init__(self, max_rpm: int = 60, max_concurrente: int = 10):
        """
        Args:
            max_rpm: máximo de requests por minuto
            max_concurrente: máximo de llamadas simultáneas
        """
        self.rate_limiter = TokenBucket(tasa=max_rpm/60, capacidad=max_rpm/60*5)
        self.processor = BatchProcessor(max_concurrente=max_concurrente)
        self.cliente: Optional[anthropic.AsyncAnthropic] = None

    async def __aenter__(self):
        self.cliente = anthropic.AsyncAnthropic().__aenter__()
        return self

    async def __aexit__(self, *args):
        if self.cliente:
            await self.cliente.__aexit__(*args)

    async def procesar_documento(self, doc: dict) -> dict:
        """Procesa un documento con rate limiting."""
        await self.rate_limiter.adquirir()
        # Aquí iría la llamada real a anthropic
        await asyncio.sleep(0.1)  # simulado
        return {
            'id': doc['id'],
            'resumen': f"Resumen de: {doc['texto'][:50]}...",
            'procesado': True
        }

    async def procesar_lote(self, documentos: list[dict]) -> ResultadoBatch:
        return await self.processor.procesar(documentos, self.procesar_documento)

# Test del pipeline
async def test_pipeline():
    docs = [{'id': i, 'texto': f'Texto del documento número {i} con contenido relevante.'}
            for i in range(20)]

    pipeline = PipelineDocumentos(max_rpm=120, max_concurrente=8)
    t0 = time.time()
    resultado = await pipeline.procesar_lote(docs)
    t_total = time.time() - t0

    print(f'Pipeline completado:')
    print(f'  Documentos: {resultado.total}')
    print(f'  Éxitos: {resultado.exitosos}')
    print(f'  Tiempo: {t_total:.2f}s')
    print(f'  Throughput: {resultado.total/t_total:.1f} docs/s')
    return resultado

resultado = asyncio.run(test_pipeline())

## Resumen

| Componente | Cuándo usar | Parámetro clave |
|---|---|---|
| `asyncio.gather` | Tareas independientes, pocos items | — |
| `asyncio.Semaphore` | Limitar concurrencia máxima | `max_concurrente` |
| `TokenBucket` | Rate limiting continuo (req/s) | `tasa`, `capacidad` |
| `BatchProcessor` | Lotes grandes con progreso | `max_concurrente` |
| `AsyncAnthropic` | Cualquier llamada a Anthropic | `timeout` |
| Streaming async | Respuestas largas, UX en tiempo real | `messages.stream()` |

**Regla práctica**: empieza con `max_concurrente=5` y `max_rpm` igual al límite de tu tier de Anthropic. Incrementa hasta que el throughput deje de mejorar o aparezcan errores 429.

### Próximos pasos
- [Gestión de errores y resiliencia](../../apis/06-gestion-errores-resilience.ipynb)
- [Batch API de Anthropic](../../apis/05-batch-api.ipynb)